In [ ]:
!pip install edgartools earningscall


In [ ]:
!pip install notebook-intelligence

In [ ]:
!pip install libsql

In [2]:
import numpy as np
import pandas as pd
import yfinance as yf
from edgar import *
from earningscall import get_company
from datetime import datetime,timedelta
import re
from libsql_client import create_client_sync

In [ ]:
conn = libsql.connect(database=TURSO_URL, auth_token=TURSO_TOKEN)
cursor = conn.cursor()

In [ ]:
client = create_client_sync()

In [ ]:
client.execute("SELECT 1")

In [ ]:
client.execute('''
    CREATE TABLE IF NOT EXISTS sec_chunks (
        chunk_id TEXT PRIMARY KEY,
        doc_id TEXT,
        ticker TEXT,
        filing_date TEXT,
        item_type TEXT,
        chunk_index INTEGER,
        content TEXT,
        sentiment_score REAL
    )
''')
print("Table 'sec_chunks' is live on Turso.")

In [3]:
set_identity("Alay Majmudar alaymajmudar.loyola@gmail.com")

In [ ]:
import re



In [ ]:
filings = Company("MSFT").get_filings(form="10-K").latest()
xbrl = filings.xbrl()
statements = xbrl.statements

# Display financial statements
balance_sheet = statements.balance_sheet()
income_statement = statements.income_statement()
cash_flow = statements.cashflow_statement()


In [ ]:
statements

In [ ]:
income_df = income_statement.to_dataframe()

In [ ]:
income_df

In [ ]:
cash_flow = cash_flow.to_dataframe()


In [ ]:
cash_flow

In [ ]:
# balance_sheet = balance_sheet.to_dataframe()
pd.set_option('display.max_rows', 110)  
balance_sheet

In [4]:
company = Company("MSFT")
    # We go back to 2018 to get a full 8 years of historical context
filings = company.get_filings(form=["10-K", "10-Q"])

In [ ]:
def get_core_financials(filing):
    """
    Extracts high-alphafinancial metrics from XBRL statements.
    """
    company = Company("MSFT")
    # We go back to 2018 to get a full 8 years of historical context
    filings = company.get_filings(form=["10-K", "10-Q"], start_date="2018-01-01")
    try:
        fin = filing.get_financials()
        
        # 1. Income Statement Data
        is_df = fin.income_statement().to_dataframe()
        # We look for the most common XBRL tags (Concepts)
        revenue = is_df[is_df['concept'].str.contains('Revenue', case=False)].iloc[0, 3]
        net_income = is_df[is_df['concept'].str.contains('NetIncome', case=False)].iloc[0, 3]

        # 2. Cash Flow Data
        cf_df = fin.cashflow_statement().to_dataframe()
        op_cash_flow = cf_df[cf_df['concept'].str.contains('Net cash from operations', case=False)].iloc[0, 2]
        capex = cf_df[cf_df['concept'].str.contains('Additions to property and equipment', case=False)].iloc[0, 2]


        #3. Balance sheet
        bal_df = fin.balance_sheet().to_dataframe()
        cash_eq = bal_df[bal_df['concept'].str.contains('Cash and cash equivalents', case=False)].iloc[0, 2]
        tot_assets = bal_df[bal_df['concept'].str.contains('Total assets', case=False)].iloc[0, 2]
        tot_liab = bal_df[bal_df['concept'].str.contains('Total liabilities', case=False)].iloc[0, 2]
        tot_eq = bal_df[bal_df['concept'].str.contains(' Total stockholders equity', case=False)].iloc[0, 2]

        return 
            ''
            "revenue": float(revenue),
            "net_income": float(net_income),
            "op_cash_flow": float(op_cash_flow),
            "capex":float(capex),
            "cash_eq":float(cash_eq),
            "total_liability":float(tot_liab),
            "total_assets":float(tot_assets),
            "total_equity":float(total_eq)
        }
    except Exception as e:
        print(f"Extraction failed for {filing.filing_date}: {e}")
        return None



In [ ]:
fin_query = [
    """
    
             CREATE TABLE IF NOT EXISTS financial_filings_raw (
        filing_id TEXT PRIMARY KEY, -- e.g., 'MSFT_2025_10K'
        ticker TEXT,
        form TEXT,           -- '10-K' or '10-Q'
        filing_date TEXT,    -- THE MOST IMPORTANT DATE (When the market saw it)
        fiscal_year INTEGER, -- e.g., 2025
        fiscal_period TEXT,  -- 'Q1', 'Q2', 'Q3', or 'FY'
        revenue REAL,
        net_income REAL,
        opcashflow REAL,            -- Operating Cash Flow
        capex REAL,          -- Additions to PPE
        casheq REAL,           -- Cash and Equivalents
        assets REAL,
        liabilities REAL,
        equity REAL
        """
]

In [ ]:
tenk = filings.obj().document
item_1a = tenk.get_section("Item 2")
if item_1a:
    # Note: Section objects also have text() method
    text = item_1a.text(include_tables=False,table_max_col_width=None)
    print(text)

In [ ]:
for f in filings:
    print(f.filing_date.isoformat().replace('-','_'))

# eightk = filings[0].obj()
# print(eightk['Item 1'])
# Get basic information
# report_date = eightk.date_of_report
# items_reported = eightk.items
# print(items_reported)

In [ ]:
'2023-3-3'.replace('-','')

In [ ]:
def clean_prose(text):
    """
    Cleans SEC prose for NLP: Normalizes whitespace and removes boilerplate headers.
    """
    if not text: return ""
    # Normalize whitespace (replaces tabs/newlines with single space)
    text = re.sub(r'\s+', ' ', text)
    # Remove standard SEC 'Table of Contents' and page markers
    text = re.sub(r'(Table of Contents|Item [0-9A-Z]+\.)', '', text, flags=re.IGNORECASE)
    return text.strip()

def get_10k_data(ticker='MSFT',years=5 ):
    company = Company('MSFT')
        # Get the last 5 10-Ks (filing once per year)
    ten_ks = company.get_filings(form="10-K").latest(years)
    data =[]
    for filing in ten_ks:
        
        f_date = filing.filing_date.isoformat()
        doc_id = f"{ticker}_10K_{f_date.replace('-','_')}"
        
        print(f"Processing 10-K filed on: {f_date}")
    
        doc = filing.obj()
        
        # Item 1: Business (Dictionary for RAG)
        raw_biz = doc['Item 1']
        clean_biz = clean_prose(raw_biz) if raw_biz else ""
    
        # Item 1A: Risk Factors (Alpha for NLP)
        raw_risk = doc["Item 1A"]
        clean_risk = clean_prose(raw_risk) if raw_risk else ""
        
        data.append({'doc_id':doc_id,'filing_date':f_date,'item1_biz':clean_biz,'item1a_risk':clean_risk})
        # Atomic Insert: Ensures 10-K data is linked to the date the market saw it
         
    print("10-K Ingestion Complete.")
    return data

In [ ]:
data = get_10k_data()

In [ ]:
def get_10q_data():
    

In [ ]:
def semantic_sentence_splitter(text, max_chars=1500):
    """Splits 'wall of text' into chunks at sentence boundaries (~350-400 tokens)."""
    if not text: return []
    # Regex split: period/exclamation/question followed by space
    sentences = re.split(r'(?<=[.!?]) +', text)
    chunks, current_chunk = [], ""

    for sentence in sentences:
        if len(current_chunk) + len(sentence) > max_chars and current_chunk:
            chunks.append(current_chunk.strip())
            current_chunk = sentence
        else:
            current_chunk += " " + sentence
    if current_chunk:
        chunks.append(current_chunk.strip())
    return chunks

In [ ]:
def sync_10k_to_turso(data_list, ticker="MSFT"):
    """
    data_list: [{'doc_id': '...', 'item1_biz': '...', 'item1a_risk': '...'}]
    """
    statements = []

    for entry in data_list:
        doc_id = entry['doc_id']
        
        filing_date = entry['filing_date']

        # Define which blobs to process
        sections = [
            ('Item 1', entry['item1_biz']),
            ('Item 1A', entry['item1a_risk'])
        ]

        for item_label, blob in sections:
            chunks = semantic_sentence_splitter(blob)
            
            for i, content in enumerate(chunks):
                # Unique PK: MSFT_20260421_ITEM1A_0
                chunk_id = f"{ticker}_{filing_date}_{item_label.replace(' ', '')}_{i}"
                
                # The SQL statement and its parameters
                sql = """
                INSERT OR REPLACE INTO sec_chunks 
                (chunk_id, doc_id, ticker, filing_date, item_type, chunk_index, content) 
                VALUES (?, ?, ?, ?, ?, ?, ?)
                """
                params = (chunk_id, doc_id, ticker, filing_date, item_label, i, content)
                statements.append((sql, params))

    # 2. Batch upload to Turso (Very efficient over the network)
    if statements:
        print(f"Preparing to sync {len(statements)} chunks to Turso...")
        print("Sample params:", statements[0][1])  
        client.batch(statements)
        print("Done! Data is now in the cloud.")
        
    else:
        print("No data found to sync.")

In [ ]:
sync_10k_to_turso(data)

In [ ]:
data

In [ ]:
modified_text = clean_mda_narrative(text)

In [ ]:
modified_text

In [ ]:
filings.obj().reports

In [ ]:
with open("./10_Q.txt", "w") as file:
    file.write(filings.obj()['Item 2'])

# filings.obj()['Item 2']

In [ ]:
for f in filings:
    print(f.parse())
    
    print(f.filing_date, f.form)

In [ ]:
sections = filings.sections()

In [ ]:
sections

In [ ]:


# Required for SEC EDGAR access


def clean_tabular_noise(text):
    """Strips lines that are likely dense tables or numeric noise."""
    if not text: return ""
    lines = str(text).split('\n')
    # Filter out lines that have 4 or more numeric/financial symbols
    cleaned_lines = [line for line in lines if len(re.findall(r'\d|\$|%', line)) < 4]
    return " ".join(cleaned_lines).strip()

def ingest_sec_data(ticker="MSFT", years=5):
    """Fetches and cleans 10-K, 10-Q, and 8-K filings."""
    end_date = datetime(2026, 4, 20)
    start_date = end_date - timedelta(days=years*365)
    date_range = f"{start_date.strftime('%Y-%m-%d')}:{end_date.strftime('%Y-%m-%d')}"
    
    company = Company(ticker)
    # Fetching 8-K is crucial for real-time volatility triggers
    filings = company.get_filings(form=["10-K", "10-Q", "8-K"], filing_date=date_range)
    
    processed_docs = []
    for f in filings:
        try:
            doc = f.obj()
            entry = {"date": f.filing_date, "form": f.form, "accession": f.accession_number}
            
            # Using bracket notation to avoid AttributeError in TenQ
            if f.form == "10-K":
                entry["mda"] = doc['Item 7']
                entry["risk_factors"] = doc['Item 1A']
                entry["business_info"] = doc['Item 1'] # Future RAG context
            elif f.form == "10-Q":
                entry["mda"] = doc['Part I, Item 2'] # Fixed key for TenQ
                entry["risk_factors"] = doc['Part II, Item 1A']
            elif f.form == "8-K":
                entry["mda"] = f.text() # 8-Ks are often unstructured press releases
            
            # Apply tabular cleaning immediately
            entry["mda_clean"] = clean_tabular_noise(entry.get("mda", ""))
            entry["risk_clean"] = clean_tabular_noise(entry.get("risk_factors", ""))
            
            processed_docs.append(entry)
        except Exception as e:
            print(f"Error parsing {f.accession_number} ({f.form}): {e}")
            
    return pd.DataFrame(processed_docs)

def ingest_market_data(ticker="MSFT", years=5):
    """Fetches daily price data and the 10-Year Treasury yield."""
    tickers = [ticker, "^TNX", "^VIX"] # MSFT, 10Y Yield, and Volatility Index
    data = yf.download(tickers, start="2021-01-01", end="2026-04-20")
    
    # Flatten multi-index columns for easier SQL ingestion
    data.columns = ['_'.join(col).strip() for col in data.columns.values]
    return data.reset_index()

In [ ]:
sec_data  = ingest_sec_data(years=1)

In [ ]:
sec_data.head()

In [ ]:
sec_data.loc[0,'mda']

In [ ]:
df_filings['risk_factors'][:1]

In [ ]:
'Q&A'.lower()

In [ ]:


company = get_company("MSFT")  # Lookup Apple, Inc by its ticker symbol, "AAPL"

transcript = company.get_transcript(year=2025, quarter=2,level=2)
print(transcript.speaker_name_map_v2)
# print(f"{company} Q3 2021 Transcript Text: \"{transcript.text[:1000]}...\"")



In [ ]:
from earningscall import get_company
def process_structured_transcript(company_ticker, year, quarter):
    company = get_company(company_ticker)
    
    # LEVEL 2 gives us the names and the text in order
    transcript = company.get_transcript(year=year, quarter=quarter, level=2)
    
    prepared_remarks = []
    qa_pairs = []
    is_qa_started = False
    qa_markers = ["q and a", "q&a","q & a"]
    
    for turn in transcript.speakers:
        speaker_name = turn.speaker_info.name.strip()
        text = turn.text.strip()

        # 1. DETECT THE PIVOT: The Operator turn that opens the floor
        if any(marker in text.lower() for marker in qa_markers):
            is_qa_started = True
            continue # Skip the operator's intro block
        
        # 2. CATEGORIZE THE TURN
        if not is_qa_started:
            prepared_remarks.append({"speaker": speaker_name, "text": text})
        else:
            qa_pairs.append({"speaker": speaker_name, "text": text})

    return prepared_remarks, qa_pairs

In [ ]:
p,q = process_structured_transcript('MSFT','2025',2)

In [ ]:
p

In [ ]:
import earningscall
from datetime import datetime, timedelta


def process_msft_history(ticker="MSFT"):
    company = earningscall.get_company(ticker)
    
    # 1. Define the 5-year boundary
    five_years_ago = datetime.now() - timedelta(days=6*365)
    
    # Storage for all processed quarters
    historical_data = []

    # 2. Iterate through events
    for event in company.events():
        # Skip future events
        if datetime.now().timestamp() < event.conference_date.timestamp():
            continue
            
        # Filter for the last 6 years
        if event.conference_date.timestamp() < five_years_ago.timestamp():
            continue

        print(f"Ingesting: Q{event.quarter} {event.year} (Date: {event.conference_date.strftime('%Y-%m-%d')})")
        
        try:
            # Fetch Level 2 to get speaker names and text
            transcript = company.get_transcript(event=event, level=2)
            
            if not transcript or not transcript.speakers:
                continue

            # 3. Apply the split logic to create our lists of dicts
            prepped, qa = split_transcript_by_turns(transcript.speakers)
            
            # Store results with metadata for the training table
            historical_data.append({
                "date": event.conference_date.strftime('%Y-%m-%d'),
                "year": event.year,
                "quarter": event.quarter,
                "prepped_list": prepped,
                "qa_list": qa
            })

        except Exception as e:
            print(f"Error processing Q{event.quarter} {event.year}: {e}")

    return historical_data

def split_transcript_by_turns(turns):
    """
    Processes Level 2 turns into two lists of dictionaries based on the Q&A pivot.
    """
    prepped_list = []
    qa_list = []
    is_qa_started = False
    
    # MSFT specific pivot markers
    qa_markers = ["q and a", "q&a"]

    for turn in turns:
        speaker_name = turn.speaker_info.name if turn.speaker else "Unknown"
        speaker_title = turn.speaker_info.title if turn.speaker else "Unknown" 
        text = turn.text if turn.text else ""
        text_lower = text.lower()

        # Detect transition
        if not is_qa_started:
            if any(marker in text_lower for marker in qa_markers):
                is_qa_started = True
                # We skip the specific turn that announces the Q&A to keep data clean
                continue 

        # Add to respective list as a dictionary
        entry = {"speaker_name": speaker_name,'speaker_title':speaker_title, "text": text}
        
        if is_qa_started:
            qa_list.append(entry)
        else:
            prepped_list.append(entry)

    return prepped_list, qa_list

# Execute
msft_dataset = process_msft_history()

In [ ]:
msft_dataset

In [ ]:
def initialize_transcript_tables(client):
    # Table A: Metadata
    client.execute('''
        CREATE TABLE IF NOT EXISTS earnings_calls (
            call_id TEXT PRIMARY KEY,
            ticker TEXT,
            date TEXT,
            year INTEGER,
            quarter INTEGER
        )
    ''')
    
    # Table B: Individual Speech Acts
    client.execute('''
        CREATE TABLE IF NOT EXISTS call_transcript_turns (
            turn_id TEXT PRIMARY KEY,
            call_id TEXT,
            section TEXT,       -- 'PREP' or 'QA'
            sequence INTEGER,   -- The order of speaking
            speaker TEXT,
            content TEXT,
            sentiment_score REAL DEFAULT 0.0
        )
    ''')
    

def sync_transcripts_to_turso(ticker, transcripts_list):
    """
    transcripts_list: Your [dict, dict, ...] from the previous step
    """
    initialize_transcript_tables()
    
    call_rows = []
    turn_rows = []
    
    for call in transcripts_list:
        call_id = f"{ticker}_{call['year']}_Q{call['quarter']}"
        
        # 1. Add to Metadata
        call_rows.append((call_id, ticker, call['date'], call['year'], call['quarter']))
        
        # 2. Add Prepared Remarks (Section: PREP)
        for i, (speaker, text) in enumerate(call['prepped_text']):
            turn_id = f"{call_id}_PREP_{i}"
            turn_rows.append((turn_id, call_id, 'PREP', i, speaker, text, 0.0))
            
        # 3. Add Q&A Section (Section: QA)
        for i, (speaker, text) in enumerate(call['qa']):
            turn_id = f"{call_id}_QA_{i}"
            turn_rows.append((turn_id, call_id, 'QA', i, speaker, text, 0.0))

    # Batch Insert (Production Style)
    cursor.executemany("INSERT OR REPLACE INTO earnings_calls VALUES (?, ?, ?, ?, ?)", call_rows)
    cursor.executemany("INSERT OR REPLACE INTO call_transcript_turns VALUES (?, ?, ?, ?, ?, ?, ?)", turn_rows)
    
    conn.commit()
    print(f"✅ Synced {len(call_rows)} calls and {len(turn_rows)} individual speech turns.")

# EXECUTE
# sync_transcripts_to_turso('MSFT', my_transcripts_list)

In [ ]:
import sys
print(sys.version)

In [ ]:
dat = yf.Ticker("MSFT")


In [ ]:
dat.info()